I'll be using this file to do some independent EDA for the retinal scan labels and retinal scan image files. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/raw/retinal_labels.csv")

In [4]:
df.head()

,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3662 entries, 0 to 3661
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id_code    3662 non-null   object
 1   diagnosis  3662 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 57.3+ KB


In [7]:
df.describe()

,diagnosis
count,3662.000000
mean,1.126980
std,1.298409
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,4.000000


In [8]:
# =============================================================================
# HINT 8: Retinal Image — BINARY Classification
# =============================================================================
# IMPORTANT: Convert the 5-class diagnosis (0-4) to BINARY for your model:
#   - 0 = No DR        (class 0 only)     — ~49% of data
#   - 1 = Has DR        (classes 1-4)      — ~51% of data
#
# This gives you an approximately 50/50 split — well balanced!
# The clinical question is: "Does this patient have ANY diabetic retinopathy?"
# This is the most useful screening question — patients with DR (any grade)
# need specialist referral, while patients without DR can wait.
# =============================================================================

def create_binary_retinal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert 5-class retinal diagnosis to binary: No DR vs Has DR.

    Raw values: 0 (No DR), 1 (Mild), 2 (Moderate), 3 (Severe), 4 (Proliferative)
    Binary target: 0 = No DR (class 0), 1 = Has DR (classes 1-4 combined)

    This gives approximately 50/50 split — well balanced for training.

    The clinical use case: screen patients to identify who needs a specialist
    referral (any DR) vs. who can wait (no DR).
    """
    df['dr_binary'] = (df['diagnosis'] > 0).astype(int)

    pos = df['dr_binary'].sum()
    neg = len(df) - pos
    print(f"Binary retinal target created:")
    print(f"  Has DR (1): {pos} ({pos/len(df)*100:.1f}%)")
    print(f"  No DR (0): {neg} ({neg/len(df)*100:.1f}%)")

    return df


def get_retinal_image_info():
    """
    Tips for working with the retinal images:

    1. Images are high-resolution PNGs — resize for training
       - Common sizes: 224x224 (ResNet), 299x299 (Inception), 384x384 (ViT)

    2. Class distribution is imbalanced:
       - Grade 0: 49% (No DR)
       - Grade 1: 10% (Mild)
       - Grade 2: 27% (Moderate)
       - Grade 3: 5%  (Severe)       <-- Very underrepresented
       - Grade 4: 8%  (Proliferative)

    3. Augmentation is critical for the minority classes:
       - Random rotation, flipping, color jitter
       - Consider: RandomResizedCrop, ColorJitter, RandomHorizontalFlip

    4. Transfer learning recommended:
       - ResNet50, EfficientNet, or DenseNet pre-trained on ImageNet
       - Fine-tune the last few layers on your retinal data
       - Freeze early layers (they detect generic features like edges)

    5. For clinical use, "referable" DR (grades 2-4) is the key metric
       - Sensitivity for referable cases is more important than overall accuracy
       - A missed severe case is much worse than a false alarm

    6. Match images to labels:
       - retinal_labels.csv has id_code and diagnosis columns
       - Image filenames in retinal_scan_images/ correspond to id_code
       - Some labels may not have matching images — filter these out
       - See match_retinal_images_to_labels() below for working join code
    """
    pass


def match_retinal_images_to_labels(labels_path: str, image_dir: str) -> pd.DataFrame:
    """
    Join retinal labels to actual image files.

    The labels file has 3,662 entries but only 3,222 images exist.
    ~440 labels have no matching image and must be filtered out.
    Always do an inner join on what's actually on disk.
    """
    import os

    labels = pd.read_csv(labels_path)
    existing_images = set(os.listdir(image_dir))

    # Keep only labels where the corresponding image file exists
    labels_matched = labels[labels['id_code'].apply(lambda x: x + '.png' in existing_images)]

    print(f"Labels in CSV: {len(labels)}")
    print(f"Images on disk: {len(existing_images)}")
    print(f"Matched (usable): {len(labels_matched)}")
    print(f"Labels without images (dropped): {len(labels) - len(labels_matched)}")

    return labels_matched
